# Investigating MNIST RLCT values using NGD and SGD

This notebook is the first in the MARS research project led by Moosa Saghir, Zach Liu, and Ragha Rao, investigating the central claim of SLT - namely that SGD converges towards sets of model parameters that yield a lower model complexity. To this end, we will use both the "real log canonical threshold" (RLCT) and the Hessian to measure model complexity.

If SLT predictions are correct, we expect that SGD will yield models with lower Hessian eigenvalues and a lower RLCT value, compared to NGD (natural gradient descent, which is a modified variant of gradient descent that premultiplies the loss gradient with the inverse of the Fisher information matrix).

## Methodology

We use a very simple network architecture:
*   128 neuron hidden layer
*   ReLU activation
*   Dropout with 0.5 probability
*   10 neuron output layer
*   Return log softmax probabilities

In future notebooks, we will also train small transformer models and CNNs. But for now we'll stick to a simple example to get started.

For different model architectures, we train the model using various gradient descent algorithms, including: SGD, Momentum, Adam, and NGD. We then calculate the Hessian and the RLCT at the minima converged to. This will allow comparison of the model complexities achieved by different gradient descent algorithms.

## Work in progress
* Need to vary model architecture and see how this affects RLCT
* Need to calculate Hessian for different optimizers / architectures

## Setup

### Linux / MacOS

These instructions will be written with Ubuntu in mind, although MacOS should be quite similar. The notebook should be run locally using VSCode, and you'll need to install Anaconda to use `conda` virtual environments (`venv`).

1. In the terminal, create a venv with a name of your choice, and activate it:

```bash
conda create --name env_name python=3.10
conda activate env_name
```

2. Install packages:

```bash
conda install einops wandb tqdm ipykernel pip
```

If some of them don't install, then try running `pip install` in the terminal. If this still doesn't work, then run `!pip install <module_name>` in the notebook itself.

We have built a custom package for this project, which can be installed as follows
```bash
pip install approxngd
```

3. Create a kernel for the virtual environment:

```bash
python -m ipykernel install --user --name env_name --display-name "Python (env_name)"
```

4. In VSCode, go to the top-right of the notebook, and select the kernel name you created. It will be in "local kernels".

5. Clone the repository into your VSCode workspace:

```
git clone https://github.com/cxtraa/ngd_with_slt.git
```

6. Remember to regularly pull/push changes using: 

``` bash
git pull origin main
git push origin main
```

7. When using `wandb`, you will be prompted for an API key. Follow the provided instructions, and you should be able to access the team "slt_to_the_moon".

In [ ]:
%pip install -r ..\requirements.txt

In [3]:
# autoreload allows refresh of submodules like PyHessian
%load_ext autoreload
%autoreload
%matplotlib inline

In [1]:
# Import required libraries

import os
import sys
import warnings
import numpy as np
import pandas as pd
import einops

sys.path.append("../")

import torch as t
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, random_split

from tqdm import tqdm
from datetime import datetime
import json
import wandb

from devinterp.slt import estimate_learning_coeff
from devinterp.slt import sample
from devinterp.slt.llc import OnlineLLCEstimator
from devinterp.slt.wbic import OnlineWBICEstimator
from devinterp.optim.sgld import SGLD

from approxngd import KFAC
from PyHessian.pyhessian import *
from PyHessian.density_plot import *
from general_utils import *
from hessian_utils import *

import plotly.express as px
import plotly.graph_objects as go
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

# device = "cpu"
device = "cuda" if t.cuda.is_available() else "cpu"
print(f"DEVICE : {device}")
warnings.filterwarnings("ignore")

c:\Users\moosa\anaconda3\envs\windows_ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DEVICE : cuda


In [21]:
# Define model hyperparameters, loss function, and optimizers

hyperparams = {
    "lr": 1e-4,
    "batch_size" : 128,
    "num_workers" : 16,
    "num_epochs" : 10,  # MUST BE AT LEAST 5 AS RLCT ESTIMATE TAKES AVERAGE OF LAST 5 EPOCHS
    "momentum" : 0.8,
    "num_draws" : 5000,
    "num_chains" : 10,
    "noise_level" : 0.0,
    "elasticity" : 1000.0,
}

epochs = np.arange(1, hyperparams["num_epochs"]+1)

criterion = {"general":nn.CrossEntropyLoss(),"kfac": nn.CrossEntropyLoss(reduction='mean')}
sgd = t.optim.SGD(
    model.parameters(),
    lr=hyperparams["lr"],
    )
adam = t.optim.Adam(
    model.parameters(),
    lr=hyperparams["lr"],
    )
rmsprop = t.optim.RMSprop(
    model.parameters(),
    lr=hyperparams["lr"],
    momentum=hyperparams["momentum"],
)
ngd = KFAC(model, 
           hyperparams["lr"], 
           1e-3,
           momentum_type='regular',
           momentum=hyperparams["momentum"],
           adapt_damping=False,
           update_cov_manually=True,
           )
optimizers = [adam]

In [11]:
# Setup hyperparameter sweep for RLCT estimation

sweep_config = {
    "method" : "random",
    "name" : "sweep",
    "parameters" : {
        "elasticity" : {
            "distribution" : "log_uniform_values",
            "min" : 1e0,
            "max" : 1e4,
        },
        "noise_level" : {
            "distribution" : "log_uniform_values",
            "min" : 1e-1,
            "max" : 1e2,
        },
        "num_draws" : {
            "distribution" : "q_uniform",
            "min" : 1e2,
            "max" : 1e3,
        },
        "num_chains" : {
            "distribution" : "int_uniform",
            "min" : 1,
            "max" : 10,
        }, 
    }, 
}

In [22]:
# Load MNIST data

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = t.utils.data.DataLoader(train_set, batch_size=hyperparams["batch_size"], shuffle=True, num_workers=hyperparams["num_workers"], persistent_workers=True)
test_loader = t.utils.data.DataLoader(test_set, batch_size=hyperparams["batch_size"], shuffle=False, num_workers=hyperparams["num_workers"], persistent_workers=True)

In [23]:
# Define training and evaluation functions

def train_one_epoch(model, train_loader, optimizer, criterion):
    """"
    Train one epoch of a model.
    `model`: the nn.Module to be trained,
    `train_loader`: the PyTorch DataLoader for the training data,
    `optimizer` : the optimizer class used,
    `criterion` : loss function.
    """
    
    model.train()
    train_loss = 0
    for image, label in tqdm(train_loader):
        image, label = image.to(device), label.to(device)
        if optimizer == ngd:
            model.zero_grad()
            # Estimate with model distribution
            with optimizer.track_forward():
                output = model(image)
                loss = criterion["kfac"](output, label)
            with optimizer.track_backward():
                loss.backward()
            optimizer.update_cov()
            # Compute loss to backprop
            model.zero_grad()
            output = model(image)
            loss = criterion["kfac"](output, label)
            train_loss += loss.item()
            loss.backward()
            optimizer.step(loss=loss)
        else:
            optimizer.zero_grad()
            output = model(image)
            loss = criterion["general"](output, label)
            train_loss += loss.item()
            loss.backward()
            optimizer.step()
    return train_loss / len(train_loader)

def evaluate(model, test_loader, criterion):
    """
    Evaluate the model with testing data.
    `model` : model to test,
    `test_loader` : PyTorch DataLoader for test data,
    `criterion` : loss function.
    """

    model.eval()
    test_loss = 0
    with t.no_grad():
        for image, label in test_loader:
            image, label = image.to(device), label.to(device)
            output = model(image)
            loss = criterion["general"](output, label)
            test_loss += loss.item()
    return test_loss / len(test_loader)


In [24]:
# For each optimiser, train the model and record train and test losses.

models = {}
train_losses = {}
test_losses = {}
for optimizer in optimizers:
    name = f"{optimizer.__class__.__name__}"
    optim_models = []
    optim_train_losses = []
    optim_test_losses = []
    print(f"\n======================== Training with {name} ==========================")
    for epoch in range(hyperparams["num_epochs"]):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        test_loss = evaluate(model, test_loader, criterion)
        optim_train_losses.append(train_loss)
        optim_test_losses.append(test_loss)
        optim_models.append(model)
        print(f"Epoch {epoch+1}/{hyperparams['num_epochs']}: train_loss={train_loss:.4f}, test_loss={test_loss:.4f}")
    train_losses[name] = optim_train_losses
    test_losses[name] = optim_test_losses
    models[name] = optim_models


======================== Training with Adam ==========================


100%|██████████| 469/469 [00:10<00:00, 43.92it/s] 


Epoch 1/10: train_loss=1.9808, test_loss=1.6548


100%|██████████| 469/469 [00:02<00:00, 232.53it/s]


Epoch 2/10: train_loss=1.4362, test_loss=1.2172


...

Epoch 8/10: train_loss=0.6428, test_loss=0.6148


100%|██████████| 469/469 [00:01<00:00, 242.57it/s]


Epoch 9/10: train_loss=0.6233, test_loss=0.5982


100%|██████████| 469/469 [00:02<00:00, 232.96it/s]


Epoch 10/10: train_loss=0.6082, test_loss=0.5844


In [25]:
# Display training and testing data and log data as a Pandas dataframe

# Training data
train_fig = go.Figure()
for optim, train_loss in train_losses.items():
    train_fig.add_trace(go.Scatter(x=epochs, y=train_loss, mode='lines+markers', name=optim))

train_fig.update_layout(title="Training loss",
                  xaxis_title="Epoch",
                  yaxis_title="Loss",
                  legend_title="Optimizers"
                  )
train_fig.show()
with open("training_data.json", "w") as file:
    json.dump(train_losses, file)
train_artifact = wandb.Artifact("training_data", type="data")
train_artifact.add_file("training_data.json")

test_fig = go.Figure()
for optim, test_loss in test_losses.items():
    test_fig.add_trace(go.Scatter(x=epochs, y=test_loss, mode='lines+markers', name=optim))

test_fig.update_layout(title="Test loss",
                  xaxis_title="Epoch",
                  yaxis_title="Loss",
                  legend_title="Optimizers",
                  )
test_fig.show()
with open("testing_data.json", "w") as file:
    json.dump(test_losses, file)
test_artifact = wandb.Artifact("testing_data", type="data")
test_artifact.add_file("testing_data.json")


ArtifactManifestEntry(path='testing_data.json', digest='8BIZRyXT7gFm4wncc5RC3w==', size=209, local_path='C:\\Users\\moosa\\AppData\\Local\\wandb\\wandb\\artifacts\\staging\\tmpug9__eya')

In [ ]:
# Log the training and testing losses to wandb

wandb.log({"Training Losses" : train_fig})
wandb.log({"Test Losses" : test_fig})

wandb.log_artifact(train_artifact)
wandb.log_artifact(test_artifact)

In [26]:
# For each model, compute the eigenspectrum of the Hessian (of final model) using the PyHessian library
# value becomes from a list of models to a single model
final_models = {key : value[-1] for key, value in models.items()}

num_batches_for_hess = 10
images, labels = [], []
iterator = iter(train_loader)
for _ in range(num_batches_for_hess):
    image, label = next(iterator)
    images.append(image)
    labels.append(label)
images = t.cat(images, dim=0)
labels = t.cat(labels, dim=0)

hessians = {}
for key, final_model in final_models.items():
    #note KFAC is used here
    crit = criterion["kfac"] if key == "KFAC" else criterion["general"]
    hessians[key] = hessian(final_model, crit, data=(images,labels), cuda=True if device=='cuda' else False)

In [27]:
# Output trace of Hessian for each optimiser

print("\n======================== HESSIAN TRACE SUMMARY BEGIN ==========================")
#trail different times to get different estimates of the trace
final_hess_means = {}
n_iters = 10
for hess in hessians:
    hess_means = []
    for _ in range(n_iters):
        hessian_trace = np.mean(hessians[hess].trace())
        hess_means.append(hessian_trace)
    final_hess_mean = np.mean(hess_means)
    final_hess_means[hess] = final_hess_mean
    print(f"Trace for {hess} : {final_hess_mean}")
print("======================= HESSIAN TRACE SUMMARY COMPLETE ========================")

with open("hessian_traces.json", "w") as file:
    json.dump(final_hess_means, file)
trace_artifiact = wandb.Artifact("hessian_traces", type="data")
trace_artifiact.add_file("hessian_traces.json")


======================== HESSIAN TRACE SUMMARY BEGIN ==========================
Trace for Adam : 126.94183812082505
======================= HESSIAN TRACE SUMMARY COMPLETE ========================


ArtifactManifestEntry(path='hessian_traces.json', digest='cGJf84YJXLz+ebtSno2DDg==', size=28, local_path='C:\\Users\\moosa\\AppData\\Local\\wandb\\wandb\\artifacts\\staging\\tmp2iy0ra2a')

In [28]:
# Produce plots of eigenspectrums and store data from eigenspectrums for integration with scipy

overlaid_fig = go.Figure()
individual_figs = []

eigenspectrum_data = {} # THIS IS WHERE THE DATA IS STORED

for key, hess in hessians.items():
    density_eigen, density_weight = hess.density()
    temp_fig = get_esd_plot_plotly(density_eigen, density_weight, title=f"{key} Hessian eigenspectrum", plot_type="log")
    temp_fig.show()
    individual_figs.append(temp_fig)

    # Assuming get_esd_plot_plotly returns a figure with one trace, add that trace to the overlaid figure
    trace=temp_fig.data[0]
    trace.name=key
    overlaid_fig.add_trace(trace)

    # save the data
    eigenspectrum_data[key] = {
        "x" : list(trace.x),
        "y" : list(trace.y),
    }

overlaid_fig.update_layout(title="Hessian eigenspectrum of optimisers",
                  xaxis_title="Eigenvalue",
                  yaxis_title="Density",
                  legend_title="Optimisers",
                  yaxis=dict(type="log"),
                  )
overlaid_fig.show()

# add overlaid fig to list of all figures
individual_figs.append(overlaid_fig)

In [30]:
# Iterate through each optimiser and integrate
# number of standard deviations for cutoff point (default is 1)

cut_offs = [1.36]

from scipy.integrate import trapz, simps

for i, (key, value) in enumerate(eigenspectrum_data.items()):

    eigenvalues = np.array(value["x"])
    density = np.array(value["y"])

    # expected eigenvalue
    mu = simps(eigenvalues*density, eigenvalues)

    # standard deviation of eigenvalues
    var = simps(eigenvalues**2 * density, eigenvalues) - mu**2
    sigma = np.sqrt(var)

    eigenvalues_small = eigenvalues[eigenvalues < cut_offs[i]]
    density_small = density[eigenvalues < cut_offs[i]]      

    area_small = simps(density_small, eigenvalues_small)
    small_eigenvalues = round(area_small * num_params)
    dimensions = num_params - small_eigenvalues

    print(f"\n====== EIGENVALUES SUMMARY: {key} ======")
    print(f"Mean: {mu} Standard deviation: {sigma}")
    print(f"Proportion of small eigenvalues: {area_small}")
    print(f"Number of small eigenvalues: {small_eigenvalues} / {num_params}")
    print(f"MODEL DIMENSONALITY ACCORDING TO HESSIAN : {dimensions}")


====== EIGENVALUES SUMMARY: Adam ======
Mean: 0.04462401454763773 Standard deviation: 0.48777355680055473
Proportion of small eigenvalues: 0.9951255644070702
Number of small eigenvalues: 3194 / 3210
MODEL DIMENSONALITY ACCORDING TO HESSIAN : 16


In [ ]:
# STANDARD RLCT MEASUREMENT WITH FIXED HYPERPARAMETERS

rlct_estimates = {}
for key, value in models.items():    
    print(f"\n======================== RLCT estimate for {key} at convergence ==========================")
    rlct_estimate = estimate_learning_coeff(
        value[-1],
        train_loader,
        criterion=criterion["general"],
        optimizer_kwargs=dict(
            lr=1e-4,
            noise_level=hyperparams["noise_level"],
            elasticity=hyperparams["elasticity"],
            num_samples=len(train_set),
            temperature="adaptive",
        ),
        sampling_method=SGLD,
        num_chains=hyperparams["num_chains"],
        num_draws=hyperparams["num_draws"],
        num_burnin_steps=0,
        num_steps_bw_draws=1,
        device=device,
    )
    rlct_estimates[key] = rlct_estimate
    print(f"RLCT estimate for {key}: {rlct_estimate}")  

In [31]:
def run_callbacks(train_loader,
                  train_data,
                  model=None,
                  callbacks=None,
                  hyperparams=None,
                  device=device,
                  criterion=None):
    
    optim_kwargs = {
        "lr" : 1e-6,
        "elasticity" : hyperparams["elasticity"],
        "temperature" : "adaptive",
        "num_samples" : len(train_data),
        "save_noise" : True,
    }

    sample(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer_kwargs=optim_kwargs,
        sampling_method=SGLD,
        num_chains=hyperparams["num_chains"],
        num_draws=hyperparams["num_draws"],
        callbacks=callbacks,
        device=device
    )

    results = {}

    for callback in callbacks:
        if hasattr(callback, "sample"):
            results.update(callback.sample())
    
    return results

In [32]:
llc_estimator = OnlineLLCEstimator(hyperparams["num_chains"],
                                   hyperparams["num_draws"], 
                                   len(train_set), 
                                   device=device)

In [15]:
noise_levels = [0.0, 1.0, 2.0, 4.0, 8.0, 16.0]
elasticities = [0.0, 10.0, 100.0, 1000.0, 10000.0, 100000.0]

rlct_figs = []

for noise_level in noise_levels:
    hyperparams['noise_level'] = noise_level
    results = run_callbacks(train_loader,
                            train_set,
                            hyperparams=hyperparams,
                            callbacks=[llc_estimator],
                            criterion=criterion["general"],
                            device=device,
                            model=model)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.arange(0,hyperparams["num_draws"]+1), y=results["llc/means"], mode='lines'))
    fig.update_layout(
        title=f"Adam RLCT estimation, Elasticity : {hyperparams['elasticity']}, Noise Level : {hyperparams['noise_level']}",
        xaxis_title="Step",
        yaxis_title="RLCT",
    )
    rlct_figs.append(fig)
    fig.show()

for elasticity in elasticities:
    hyperparams['elasticity'] = elasticity
    results = run_callbacks(train_loader,
                        train_set,
                        weights=models["Adam"][-1].parameters(),
                        hyperparams=hyperparams,
                        callbacks=[llc_estimator])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.arange(0,hyperparams["num_draws"]+1), y=results["llc/means"], mode='lines'))
    fig.update_layout(
        title=f"Adam RLCT, Elasticity : {hyperparams['elasticity']}, Noise Level : {hyperparams['noise_level']}",
        xaxis_title="Step",
        yaxis_title="RLCT",
    )
    rlct_figs.append(fig)
    fig.show()

Chain 0:  17%|█▋        | 867/5000 [00:03<00:14, 281.58it/s]


KeyboardInterrupt: 

In [33]:
results = run_callbacks(train_loader,
                        train_set,
                        model,
                        callbacks=[llc_estimator],
                        hyperparams=hyperparams,
                        device=device,
                        criterion=criterion["general"])

Chain 9: 100%|██████████| 5000/5000 [00:15<00:00, 315.23it/s]


In [34]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(0,hyperparams["num_draws"]+1), y=results["llc/means"], mode='lines'))
fig.update_layout(
    title="LLC estimations wrt. steps for Adam without ReLU (DLN)",
    xaxis_title="Step",
    yaxis_title="RLCT",
)
fig.show()

In [ ]:
# Export all figures to a html

write_figs_to_html(rlct_figs, dest="elasticity-noise-level.html", title="Convergence of RLCT value for different elasticities and noise levels")

In [ ]:
# HYPERPARAMETER SWEEP FOR RLCT MEASUREMENTS

def wandb_rlct_estimation():
    """
    Used in hyperparameter sweep.
    Estimate RLCT for the first optimizer only using random hyperparameters for SGLD.
    """
    rlct_estimates = {}
    with wandb.init() as run:
        config = run.config
        for optimizer in ["Adam"]:    
            print(f"======================== RLCT estimates for {optimizer} ==========================")
            for m in models[optimizer]:
                rlct_estimate = estimate_learning_coeff(
                    m,
                    train_loader,
                    criterion=criterion["general"],
                    optimizer_kwargs=dict(
                        lr=hyperparams["lr"],
                        noise_level=config.noise_level,
                        elasticity=config.elasticity,
                        num_samples=len(train_set),
                        temperature="adaptive",
                    ),
                    sampling_method=SGLD,
                    num_chains=config.num_chains,
                    num_draws=config.num_draws,
                    num_burnin_steps=0,
                    num_steps_bw_draws=1,
                    device=device,
                )
                if optimizer in rlct_estimates:
                    rlct_estimates[optimizer].append(rlct_estimate)
                else:
                    rlct_estimates[optimizer] = [rlct_estimate]
            average_rlct_estimate = np.sum(rlct_estimates[optimizer][-5:])/5        
            wandb.log({"optimizer" : optimizer, "rlct_estimate" : average_rlct_estimate})
            print(f"###### FINAL RLCT ESTIMATE FOR {optimizer} : {average_rlct_estimate} ######")

In [ ]:
# RUN HYPERPARAMETER SWEEP

hyperparam_sweep_2 = wandb.sweep(sweep_config, project=PROJECT_NAME)
wandb.agent(hyperparam_sweep_2, wandb_rlct_estimation)

In [ ]:
import numpy as np
a = np.ones(100)


In [ ]:
# Save RLCT estimates data to wandb

rlct_fig = go.Figure()

for optim, rlct_estimate in rlct_estimates.items():
    rlct_fig.add_trace(go.Scatter(x=epochs, y=rlct_estimate, mode='lines+markers', name=optim))

rlct_fig.update_layout(title="RLCT estimate",
                  xaxis_title="Epoch",
                  yaxis_title="RLCT",
                  legend_title="Optimizers",
                  )

wandb.log({"RLCT Estimates" : rlct_fig})

In [ ]:
# Quick graphs to visualise how RLCT evolved over time

#plt.figure(figsize=(10, 6))
for optim in rlct_estimates:
    #data = {"Epochs" : np.arange(1, hyperparams["num_epochs"]+1), optim : rlct_estimates[optim]}
    plt.plot(np.arange(1, hyperparams["num_epochs"]+1), rlct_estimates[optim], label=optim)
plt.grid()
plt.title("RLCT estimation vs. epochs")
plt.xlabel("Epoch")
plt.ylabel("RLCT")
plt.legend()
plt.show()